# CareLoop Clinics: Post-Visit Call Note Root-Cause Extraction
**Objective:** Use structured LLM extraction (`google-genai`) paired with strict Pydantic schema enforcement to categorize unstructured front-desk follow-up notes into actionable operational root causes.

In [ ]:
import json
import os
import time
from typing import Literal
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

API_KEY = os.environ.get("CARELOOP_CLINIC_KEY") or os.environ.get("careloop_clinic_key")
client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-2.5-flash"

### 1. Define Structured Output Schema & Pydantic Contract
We enforce both API-level JSON schema constraints and client-side Pydantic validation (`Literal` taxonomy + bounded confidence score) so malformed outputs are caught and quarantined cleanly.

In [ ]:
CauseCategory = Literal[
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

ALLOWED_CAUSES = list(CauseCategory.__args__)

class CallNoteAudit(BaseModel):
    root_cause: CauseCategory
    confidence: float = Field(ge=0.0, le=1.0)
    summary: str = Field(min_length=5, max_length=160)
    recommended_action: str = Field(min_length=5)

SYSTEM_PROMPT = (
    "You are a clinical operations analyst for CareLoop Clinics. "
    "Classify missed-appointment follow-up call notes strictly into the allowed taxonomy, "
    "assigning 'other' whenever a note is vague or multi-causal."
)

def classify_call_note(raw_note: str) -> dict:
    user_prompt = (
        f"Classify this post-appointment call note into one of {ALLOWED_CAUSES}. "
        f"Include a confidence score (0-1), a concise summary under 25 words, "
        f"and one concrete front-desk follow-up action.\n\nCall Note: \"{raw_note}\""
    )
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.0,
            max_output_tokens=512,
            response_mime_type="application/json",
            response_schema=CallNoteAudit,
        ),
    )
    return json.loads(response.text)

# Smoke test on a single note
sample_audit = classify_call_note("Forgot about the appointment. No reminder SMS received.")
print("Smoke Test Output:\n", json.dumps(sample_audit, indent=2))

### 2. Batch Audit & Cross-Tabulation Against Reminder Logs
We process a 10-note audit batch with blank-note handling and rate-limit pacing, then join the structured labels against `appointments_clean.csv` to verify whether reminder-related misses align with actual SMS/call logs.

In [ ]:
notes_df = pd.read_csv("data/followup_notes.csv")
audit_rows = []
requests_sent = 0

for _, row in notes_df.head(10).iterrows():
    n_id = row["note_id"]
    a_id = row["appointment_id"]
    text = row["note_text"]

    if pd.isna(text) or not str(text).strip():
        audit_rows.append({
            "note_id": n_id,
            "appointment_id": a_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

    requests_sent += 1
    raw_payload = ""
    try:
        parsed_json = classify_call_note(str(text).strip())
        raw_payload = json.dumps(parsed_json)
        validated = CallNoteAudit(**parsed_json)

        audit_rows.append({
            "note_id": n_id,
            "appointment_id": a_id,
            "root_cause": validated.root_cause,
            "confidence": validated.confidence,
            "summary": validated.summary,
            "recommended_action": validated.recommended_action,
            "validation_status": "valid",
            "raw_response": raw_payload,
        })
    except (ValidationError, ValueError, json.JSONDecodeError) as err:
        fallback = json.loads(raw_payload) if raw_payload else {}
        audit_rows.append({
            "note_id": n_id,
            "appointment_id": a_id,
            "root_cause": fallback.get("root_cause", ""),
            "confidence": fallback.get("confidence", None),
            "summary": fallback.get("summary", ""),
            "recommended_action": fallback.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_payload,
        })

    time.sleep(13)

analysed_df = pd.DataFrame(audit_rows)
analysed_df.to_csv("notes_analysed.csv", index=False)
print(f"Batch complete. API calls executed: {requests_sent}")

# Cross-tabulate extracted root causes against historical reminder_sent status
clean_appts = pd.read_csv("appointments_clean.csv")
joined_audit = pd.merge(analysed_df, clean_appts, on="appointment_id", how="inner")

print("\n--- Root Cause vs. Prior Reminder Sent ---")
print(pd.crosstab(joined_audit["root_cause"], joined_audit["reminder_sent"]))